# Ingesta Crisol Librería

**Objetivo**: scrapear listados de productos desde crisol.com.pe, guardar un snapshot crudo + una versión deduplicada y subir a HDFS.

## Dependencias

In [1]:
# %pip install --quiet requests beautifulsoup4 lxml pandas numpy

In [2]:
import sys,bs4, requests
print(sys.executable)
print(bs4.__version__, requests.__version__)

/home/bigdata/venvs/pyspark_env/bin/python
4.14.2 2.32.3


## Parámetros

In [3]:
from datetime import datetime, date
import os, time, random

print("=== INGESTA CRISOL — ONE SHOT ===")
print("Inicio:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

RAW_DAY   = date.today().isoformat()
LOCAL_DIR = "/srv/bigdata"         
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"RAW_DAY  : {RAW_DAY}")
print(f"LOCAL_DIR: {LOCAL_DIR}")

=== INGESTA CRISOL — ONE SHOT ===
Inicio: 2025-10-20 15:26:34
RAW_DAY  : 2025-10-20
LOCAL_DIR: /srv/bigdata


## Conectividad

In [4]:
import requests

BASE = "https://www.crisol.com.pe"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/120.0.0.0 Safari/537.36")
HEADERS = {"User-Agent": USER_AGENT}
REQUEST_TIMEOUT = 20

print("Chequeando HOME…", BASE, end="  ")
r = requests.get(BASE, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
print("OK" if r.ok else f"FALLA ({r.status_code})")

Chequeando HOME… https://www.crisol.com.pe  OK


## Utilidades

In [5]:
import re
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

def clean_money(s):
    if not s: return None
    s = (s.replace("\u00a0"," ")
           .replace("S/.", "")
           .replace("S/", "")
           .replace("S/ ", "")
           .strip())
    digits = "".join(ch for ch in s if ch.isdigit() or ch in ".,")
    digits = digits.replace(",", "")
    try: return float(digits)
    except: return None

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

# Selectores flexibles (Crisol suele ser ecommerce con grillas tipo Magento/VTEX)
SELECTORS = {
    "product_card": [
        "li.product", "li.product-item", "div.product-item",
        "div.product.product-item", "div.product-grid-item",
        "ol.products li", "ul.products li", "div.products div.product"
    ],
    "title": [
        "a.product-item-link", "h2.product.name a", "h3.product-title a", "a.product.name",
        "a.product-title", "h2.product-title a"
    ],
    "price_current": [
        "span.price-final_price span.price", "span.special-price span.price",
        "span.price-wrapper span.price", "span.price", "span.product-price"
    ],
    "price_old": [
        "span.old-price span.price", "del span.price",
        "span.price-wrapper .old-price span.price", "del.price", "span.list-price"
    ],
    "link": [
        "a.product-item-link", "h2.product.name a", "a.product.name", "h3.product-title a",
        "a.product-title"
    ],
    "next_page": [
        "a[title='Siguiente']", "a.next", "a[rel='next']",
        "li.pages-item-next a", "a.action.next", "a.pagination-next"
    ],
}

def first_text(el, candidates):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get_text(strip=True):
            return f.get_text(strip=True)
    return ""

def first_attr(el, candidates, attr="href"):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get(attr):
            return f.get(attr)
    return ""

def url_to_catname(u: str) -> str:
    p = urlparse(u).path.strip("/").replace(".html","")
    parts = [x for x in p.split("/") if x]
    return " / ".join([x.replace("-", " ").title() for x in parts[:4]]) or "General"

## Taxonomía

In [6]:
## Taxonomia

import unicodedata, json

taxonomy = {
  "taxonomy_version": "2025-09-screenshots",
  "top": ["Los más vendidos","Novedades","Ficción","No ficción","Infantil","Juvenil","Smart Play","Tecnología"],
  "Los más vendidos": {"type": "collection", "children": ["Los más vendidos del año","Los 100 más vendidos del mes"]},
  "Novedades": {"type": "collection", "children": ["Nuevos lanzamientos","Preventa"]},
  "Ficción": {"groups": {
      "Resaltados (colecciones)": ["Voces Peruanas","Vargas Llosa","Premios","Juego de tronos","Star wars"],
      "Narrativa": ["Literatura clásica","Literatura Latinoamericana","Literatura Peruana","Literatura Universal","Literatura en otros idiomas","Literatura de viajes"],
      "Novelas": ["Ciencia ficción","Fantasía","Novela histórica","Novela negra","Romántica y erótica","Terror"],
      "Lírica y drama": ["Poesía","Teatro"],
      "Mangas": ["Adulto","Seinen","Shojo","Shonen"],
      "Cómics y novela gráfica": ["Ciencia ficción, fantasía y horror","Clásicos de la literatura","Cómic de héroes","Cómic infantil","Erótico","Cómic Europeo","Humor","Cómics en inglés","Novela gráfica","Cómic Americano","Libros de ilustración","Cómic Peruano"]
  }},
  "No ficción": {"groups": {
      "Resaltados (colecciones)": ["Los más vendidos","Actualidad Peruana","Artículos del mes","Rosa María Cifuentes","Gift cards"],
      "Libros de no ficción": ["Ciencias","Humanidades","Ciencias sociales","Desarrollo personal y salud","Empresa"],
      "Artes e ilustrados / Escolar / Técnicos": ["Artes e ilustrados","Escolar y plan lector","Gastronomía","Libro técnico","Deportes"],
      "Consulta y otros": ["Obras de consulta","Turismo","Miscelánea","Enseñanza de idiomas"]
  }, "deep": { "Humanidades": ["Biografías","Comunicación","Educación","Educación en el Perú","Ensayo","Estudios literarios y lingüística","Estudios literarios y lingüística del Perú","Filosofía","Geografía","Historia del Perú","Mitología","Psicoanálisis","Psicología","Religión"]}},
  "Infantil": {"groups": {
      "De 0 a 2 años": ["Cuentos imprescindibles","Hábitos y emociones","Libros con solapa","Primeros conocimientos","Tus personajes favoritos"],
      "De 3 a 5 años": ["Álbumes ilustrados","Cuentos clásicos","Libros para colorear","Prelectura y escritura","Primeros conocimientos","Tus personajes favoritos","Valores"],
      "De 6 a 8 años": ["Literatura de 6 a 8 años","Álbumes ilustrados","Ciencia","Cultura y sociales","Libros con valores","Libros para colorear","Libros que te conectan a la lectura","Manualidades","Primeros idiomas","Tus personajes favoritos"],
      "De 9 a 12 años": ["Literatura de 9 a 12","Ciencia","Cocina y manualidades","Cultura y sociales"]
  }},
  "Juvenil": {"groups": {
      "Wattpad": ["Wattpad"],
      "Romance spicy": ["Romance"],
      "Comics y mangas": ["Shojo","Shonen","Comics de héroes","One piece"],
      "Literatura juvenil": ["Terror y misterio","Fantasía","Ciencia ficción","Youtubers","Drama","Clásicos"],
      "Fantasía": ["Fantasía"]
  }},
  "Smart Play": {"groups": {
      "Resaltados (colecciones)": ["Los más vendidos","Novedades","Mundo Harry Potter","Productos Steam","Artículos del mes","Gift cards"],
      "Smart play": ["Rompecabezas","Juguetes educativos","Juegos de mesa","Vinilos"],
      "Complementos": ["Merchandising","Agendas y libretas","Útiles de escritorio"]
  }},
  "Tecnología": {"groups": {
      "Audífonos": ["Audífonos deportivos/open ear","Audífonos headphones/on ear","Audífonos in ear","Audífonos inalámbricos"],
      "Cargadores": ["Cargadores para auto","Cargadores de carga rápida","Cargadores portátiles","Cables de carga"],
      "Parlantes & Altavoces": ["Parlantes","Parlantes mini"]
  }}
}

In [7]:
def slugify(name: str) -> str:
    s = unicodedata.normalize("NFKD", name).encode("ascii","ignore").decode("ascii")
    s = s.lower()
    s = s.replace("&"," y ").replace("/", " ").replace(".", " ")
    s = re.sub(r"\s+y\s+"," y ", s)
    s = re.sub(r"[^a-z0-9\s-]","", s)
    s = re.sub(r"\s+","-", s).strip("-")
    return s

def flatten_taxonomy(obj):
    """Aplana el JSON en una lista de 'rutas' candidatas (en texto)"""
    paths = set()
    def add(label, prefix=""):
        p = "/".join(x for x in [prefix, slugify(label)] if x)
        paths.add(p)
        return p
    # top
    for t in taxonomy.get("top", []):
        add(t)
    # niveles conocidos
    for key, val in taxonomy.items():
        if isinstance(val, dict):
            if "children" in val:
                base = add(key)
                for c in val["children"]:
                    add(c, base)
            if "groups" in val:
                base = add(key)
                for grp, items in val["groups"].items():
                    g = add(grp, base)
                    for it in items:
                        add(it, g)
            if "deep" in val:
                base = add(key)
                for grp, items in val["deep"].items():
                    g = add(grp, base)
                    for it in items:
                        add(it, g)
    return sorted(paths)

candidates = flatten_taxonomy(taxonomy)

def check_url_variants(path_slug):
    """Prueba URL con y sin .html (algunos sitios usan ambas formas)."""
    variants = [
        f"{BASE}/{path_slug}.html",
        f"{BASE}/{path_slug}",
    ]
    for u in variants:
        try:
            r = requests.get(u, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code == 200:
                return u
        except Exception:
            pass
    return None

CATEGORIES = []
for p in candidates:
    u = check_url_variants(p)
    if u: CATEGORIES.append(u)

print("=== Categorías validadas (Crisol) ===")
print("Total:", len(CATEGORIES))
for u in CATEGORIES[:12]:
    print(" -", u)

assert CATEGORIES, "No se validó ninguna categoría de Crisol; revisa estructura/slug/robots."

=== Categorías validadas (Crisol) ===
Total: 67
 - https://www.crisol.com.pe/ficcion
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/ciencia-ficcion-fantasia-y-horror
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/clasicos-de-la-literatura
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/comic-de-heroes
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/comic-infantil
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/erotico
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/humor
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/libros-de-ilustracion
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/novela-grafica
 - https://www.crisol.com.pe/ficcion/lirica-y-drama
 - https://www.crisol.com.pe/ficcion/lirica-y-drama/poesia


## Parser de listados

In [8]:
def parse_listing(html: str, base_url: str):
    soup = BeautifulSoup(html, "lxml")

    # Tarjetas
    cards = []
    for sel in SELECTORS["product_card"]:
        cards = soup.select(sel)
        if cards: break

    rows = []
    for card in cards:
        title = first_text(card, SELECTORS["title"])
        link  = first_attr(card, SELECTORS["link"], "href")
        pcur  = first_text(card, SELECTORS["price_current"])
        pold  = first_text(card, SELECTORS["price_old"])

        rows.append({
            "titulo": title,
            "autor": "",  # opcional: enriquecer en otra pasada
            "precio_actual":  clean_money(pcur),
            "precio_anterior": clean_money(pold),
            "url_producto": urljoin(base_url, link) if link else "",
        })

    # Siguiente página
    next_url = ""
    for sel in SELECTORS["next_page"]:
        a = soup.select_one(sel)
        if a and a.get("href"):
            next_url = urljoin(base_url, a.get("href"))
            break

    return rows, next_url

## Crawl multi-categoría

In [9]:
from datetime import datetime
import pandas as pd

SLEEP_RANGE = (1.5, 3.0)   # respeta al sitio
MAX_PAGES   = 60           # tope de seguridad

def crawl_category(cat_url, max_pages=MAX_PAGES, sleep_range=SLEEP_RANGE):
    url   = cat_url
    page  = 1
    data  = []
    seen  = set()

    while url and page <= max_pages:
        if url in seen:
            break
        seen.add(url)

        try:
            html = fetch(url)
        except Exception as e:
            print("[error]", e, "->", url)
            break

        items, next_url = parse_listing(html, url)
        for it in items:
            it["categoria"] = url_to_catname(cat_url)
        data.extend(items)

        print(f"[info] {url_to_catname(cat_url)} | página {page}: {len(items)} items")
        url  = next_url
        page += 1
        time.sleep(random.uniform(*sleep_range))

    return data

# === correr ===
all_rows = []
t0 = datetime.now()
print("\n=== INICIANDO CRAWL ===")

for i, cat in enumerate(CATEGORIES, 1):
    t_cat = datetime.now()
    print(f"[{i}/{len(CATEGORIES)}] {cat}")
    data_cat = crawl_category(cat)
    all_rows.extend(data_cat)
    print(f" -> items categ.: {len(data_cat)} | acumulado: {len(all_rows)} | t: {datetime.now()-t_cat}")
    print("-"*60)

print("=== FIN CRAWL ===")
print("Total filas crudas:", len(all_rows))
print("Tiempo total:", datetime.now()-t0)

df = pd.DataFrame(
    all_rows,
    columns=["titulo","autor","precio_actual","precio_anterior","url_producto","categoria"]
)
df.insert(0, "fecha_scraping", datetime.now().strftime("%Y-%m-%d"))
print("Preview:")
df.head(3)


=== INICIANDO CRAWL ===
[1/67] https://www.crisol.com.pe/ficcion
[info] Ficcion | página 1: 20 items
[info] Ficcion | página 2: 20 items
[info] Ficcion | página 3: 20 items
[info] Ficcion | página 4: 20 items
[info] Ficcion | página 5: 20 items
[info] Ficcion | página 6: 20 items
[info] Ficcion | página 7: 20 items
[info] Ficcion | página 8: 20 items
[info] Ficcion | página 9: 20 items
[info] Ficcion | página 10: 20 items
[info] Ficcion | página 11: 20 items
[info] Ficcion | página 12: 20 items
[info] Ficcion | página 13: 20 items
[info] Ficcion | página 14: 20 items
[info] Ficcion | página 15: 20 items
[info] Ficcion | página 16: 20 items
[info] Ficcion | página 17: 20 items
[info] Ficcion | página 18: 20 items
[info] Ficcion | página 19: 20 items
[info] Ficcion | página 20: 20 items
[info] Ficcion | página 21: 16 items
[info] Ficcion | página 22: 20 items
[info] Ficcion | página 23: 20 items
[info] Ficcion | página 24: 20 items
[info] Ficcion | página 25: 19 items
[info] Ficcion | p

,fecha_scraping,titulo,autor,precio_actual,precio_anterior,url_producto,categoria
0,2025-10-20,Nuestra Señora de París (edición ilustrada),,8330.0,11900.0,https://www.crisol.com.pe/libro-nuestra-senora...,Ficcion
1,2025-10-20,El último secreto. Robert Langdon. 6,,7992.0,9990.0,https://www.crisol.com.pe/libro-el-ultimo-secr...,Ficcion
2,2025-10-20,La profesora,,7120.0,8900.0,https://www.crisol.com.pe/libro-la-profesora-9...,Ficcion


In [11]:
def fix_after(df, col):
    s = df[col].astype(str)
    df[col] = pd.to_numeric(
        np.where(s.str.fullmatch(r"\d+"), s, s), errors="coerce"
    )
    df[col] = np.where(df[col] >= 1000, df[col] / 100.0, df[col])
    return df

import numpy as np
df = fix_after(df, "precio_actual")
df = fix_after(df, "precio_anterior")
df

,fecha_scraping,titulo,autor,precio_actual,precio_anterior,url_producto,categoria
0,2025-10-20,Nuestra Señora de París (edición ilustrada),,83.30,119.0,https://www.crisol.com.pe/libro-nuestra-senora...,Ficcion
1,2025-10-20,El último secreto. Robert Langdon. 6,,79.92,99.9,https://www.crisol.com.pe/libro-el-ultimo-secr...,Ficcion
2,2025-10-20,La profesora,,71.20,89.0,https://www.crisol.com.pe/libro-la-profesora-9...,Ficcion
3,2025-10-20,Unos cuantos sueños,,79.20,99.0,https://www.crisol.com.pe/libro-unos-cuantos-s...,Ficcion
4,2025-10-20,Orgullo y prejuicio,,31.92,39.9,https://www.crisol.com.pe/libro-orgullo-y-prej...,Ficcion
...,...,...,...,...,...,...,...
18299,2025-10-20,Rompecabezas 3D National Geographic Dinosaur Park,,69.01,NaN,https://www.crisol.com.pe/rompecabezas-6944588...,Smart Play / Smart Play / Rompecabezas
18300,2025-10-20,Rompecabezas 3D Notre Dame De Paris Led,,109.00,NaN,https://www.crisol.com.pe/rompecabezas-6944588...,Smart Play / Smart Play / Rompecabezas
18301,2025-10-20,Rompecabezas 3D St. Basil'S Cathedral Led,,109.00,NaN,https://www.crisol.com.pe/rompecabezas-6944588...,Smart Play / Smart Play / Rompecabezas
18302,2025-10-20,Rompecabezas 3D Eiffel Tower Led,,149.00,NaN,https://www.crisol.com.pe/rompecabezas-6944588...,Smart Play / Smart Play / Rompecabezas


## Guardar SNAPSHOT

In [12]:
TS = datetime.now().strftime("%Y%m%d_%H%M%S")
SNAPSHOT_OUT = os.path.join(LOCAL_DIR, f"crisol_precios_allcats_{TS}.csv")
df.to_csv(SNAPSHOT_OUT, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(SNAPSHOT_OUT)/1024/1024
print(f"[OK] Guardado crudo: {SNAPSHOT_OUT} ({size_mb:.2f} MB)")

[OK] Guardado crudo: /srv/bigdata/crisol_precios_allcats_20251020_203525.csv (2.66 MB)


## Verificaciones

In [13]:
print("Nulos por columna:"); print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicados por url_producto:", df.duplicated("url_producto").sum())
print("\nTop categorías:"); print(df["categoria"].value_counts().head(10))

Nulos por columna:
precio_anterior    1738
titulo                0
fecha_scraping        0
autor                 0
precio_actual         0
url_producto          0
categoria             0
dtype: int64

Duplicados por url_producto: 7125

Top categorías:
categoria
No Ficcion                           1199
Infantil                             1198
No Ficcion / Humanidades             1197
Ficcion / Narrativa                  1193
Juvenil                              1190
Ficcion                              1185
Ficcion / Mangas                      958
Juvenil / Literatura Juvenil          785
Ficcion / Novelas                     747
Ficcion / Comics Y Novela Grafica     602
Name: count, dtype: int64


## Deduplicado por URL normalizada

In [14]:
url = df["url_producto"].astype(str).str.strip()
url_norm = (url.str.lower()
              .str.replace(r"#.*$", "", regex=True)
              .str.replace(r"\?.*$", "", regex=True)
              .str.rstrip("/"))
df["url_norm"] = url_norm

print("Productos (url_norm) únicos antes de dedup:", df["url_norm"].nunique())

catmap = (df.assign(categoria=df["categoria"].astype(str).str.strip())
            .groupby("url_norm")["categoria"]
            .apply(lambda s: " | ".join(sorted(set([c for c in s if c]))))
            .reset_index(name="categorias_join"))

df["_score"] = df["precio_actual"].notna().astype(int)
keep = (df.sort_values(["url_norm","fecha_scraping","_score"], ascending=[True, False, False])
          .drop_duplicates("url_norm", keep="first")
          .drop(columns=["_score"])
          .copy())

keep = keep.merge(catmap, on="url_norm", how="left")
cols = ["fecha_scraping","titulo","autor","precio_actual","precio_anterior",
        "url_producto","categoria","categorias_join","url_norm"]
keep = keep[cols]

DEDUP_OUT = os.path.join(LOCAL_DIR, "crisol_snapshot_dedup.csv")
keep.to_csv(DEDUP_OUT, index=False, encoding="utf-8-sig")

print("Filas originales:", len(df))
print("Productos únicos:", keep["url_norm"].nunique())
print("[OK] Guardado dedup:", DEDUP_OUT)## Subir HDFS

Productos (url_norm) únicos antes de dedup: 11179
Filas originales: 18304
Productos únicos: 11179
[OK] Guardado dedup: /srv/bigdata/crisol_snapshot_dedup.csv


## Enriquecimiento desde cada producto

In [23]:
# --- Bloque de arranque seguro para ENRIQUECIMIENTO ---

import pandas as pd
import os

# 0) Si el DataFrame 'keep' no está en memoria, cárgalo del CSV deduplicado
#    Ajusta la ruta si tu archivo se llama distinto
DEDUP_PATH = os.path.join(LOCAL_DIR, "crisol_snapshot_dedup.csv")
if "keep" not in globals():
    assert os.path.exists(DEDUP_PATH), f"No encuentro el dedup CSV: {DEDUP_PATH}"
    keep = pd.read_csv(DEDUP_PATH)
    print("[info] Cargado keep desde CSV:", DEDUP_PATH, "| filas:", len(keep))

# 1) Asegura que existan TODAS las columnas que usará el enriquecimiento
TARGET_COLS = ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]
for c in TARGET_COLS:
    if c not in keep.columns:
        keep[c] = ""   # crea vacía (string) para evitar KeyError

# 2) (opcional) Asegura también columnas base usadas en filtros
BASE_NEEDS = ["url_producto","categoria","categorias_join"]
for c in BASE_NEEDS:
    if c not in keep.columns:
        keep[c] = ""

print("[ok] Columnas presentes:", [c for c in TARGET_COLS if c in keep.columns])

[ok] Columnas presentes: ['autor', 'isbn', 'sku', 'editorial', 'edicion', 'anio_edicion', 'paginas', 'product_id']


In [24]:
import re, json, time, random
from bs4 import BeautifulSoup
import requests
from datetime import datetime

ENRICH_LIMIT  = None          # None = todos los candidatos; o pon un número para pruebas
ENRICH_SLEEP  = (0.8, 1.6)    # respeta al sitio

# --------- 0) ¿Qué filas enriquecer? ----------
# a) URLs válidas
mask_urls_ok = keep["url_producto"].astype(str).str.len().gt(0)

# b) Falta alguno de los campos que queremos poblar
needs_any = (
    keep["autor"].astype(str).str.strip().eq("") |
    keep["isbn"].astype(str).str.strip().eq("") |
    keep["editorial"].astype(str).str.strip().eq("") |
    keep.get("edicion",   pd.Series([""]*len(keep))).astype(str).str.strip().eq("") |
    keep.get("anio_edicion", pd.Series([""]*len(keep))).astype(str).str.strip().eq("") |
    keep.get("paginas",   pd.Series([""]*len(keep))).astype(str).str.strip().eq("") |
    keep.get("product_id",pd.Series([""]*len(keep))).astype(str).str.strip().eq("")
)

# c) Priorizamos lo que se parece a libro (mejora el % de ISBN)
mask_cat_libros = (
    keep["categoria"].astype(str).str.contains("libro", case=False, na=False) |
    keep["categorias_join"].astype(str).str.contains("libro", case=False, na=False)
)
mask_url_isbn = keep["url_producto"].astype(str).str.contains(r"(?:97[89][-\d]{8,}|isbn)", case=False, na=False)

pending_mask = mask_urls_ok & (needs_any | mask_cat_libros | mask_url_isbn)

to_enrich = (
    keep.loc[pending_mask, ["url_producto"]]
        .drop_duplicates()
        .reset_index(drop=True)
)
if ENRICH_LIMIT:
    to_enrich = to_enrich.head(ENRICH_LIMIT)

print("Candidatos a enriquecer:", len(to_enrich))
display(to_enrich.head(3))

Candidatos a enriquecer: 11174


,url_producto
0,https://www.crisol.com.pe/12-reglas-para-vivir...
1,https://www.crisol.com.pe/2025-family-desk-pad...
2,https://www.crisol.com.pe/24-colores-super-int...


In [25]:
# --------- 1) Extractor por página ---------
def only_digits(s):
    return re.sub(r"[^\d]", "", s or "")

def extract_meta_from_product(url: str) -> dict:
    """
    Devuelve: autor, isbn, sku, editorial, edicion, anio_edicion, paginas, product_id
    de una página de producto Crisol.
    """
    meta = {
        "autor": "", "isbn": "", "sku": "", "editorial": "",
        "edicion": "", "anio_edicion": "", "paginas": "", "product_id": ""
    }
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        if r.status_code != 200:
            return meta
    except Exception:
        return meta

    soup = BeautifulSoup(r.text, "lxml")

    # 1) Tabla de especificaciones (Magento)
    for tr in soup.select("table.data.table.additional-attributes tr"):
        th = tr.select_one("th, td[data-th]")
        td = tr.select_one("td")
        if not th or not td: 
            continue
        label = th.get_text(" ", strip=True).lower()
        value = td.get_text(" ", strip=True)

        if "autor" in label and not meta["autor"]:
            meta["autor"] = value
        if "isbn" in label and not meta["isbn"]:
            meta["isbn"] = only_digits(value)  # deja solo dígitos
        if "editorial" in label and not meta["editorial"]:
            meta["editorial"] = value
        if "edición" in label and not meta["edicion"]:
            meta["edicion"] = only_digits(value)  # suele ser “1”, “2”, etc.
        if "año de edición" in label and not meta["anio_edicion"]:
            meta["anio_edicion"] = only_digits(value)
        if "páginas" in label and not meta["paginas"]:
            meta["paginas"] = only_digits(value)

    # 2) SKU por fuera
    cand = soup.select_one("div.product.attribute.sku .value, span.sku, div.sku .value")
    if cand and not meta["sku"]:
        meta["sku"] = cand.get_text(strip=True)

    # 3) Product ID escondido en inputs/metas/data-attrs (fallback)
    #    Tomamos el primer group de 6+ dígitos como id de producto si aparece.
    html = soup.decode()
    m = re.search(r'product[_\- ]?id["\']?\s*[:=]\s*["\']?(\d{6,})', html, re.I)
    if not m:
        m = re.search(r'data-product-id=["\'](\d{6,})', html, re.I)
    if not m:
        m = re.search(r'\bproductId\b["\']?\s*[:=]\s*["\']?(\d{6,})', html, re.I)
    if m:
        meta["product_id"] = m.group(1)

    # 4) Fallback ISBN en todo el texto si aún está vacío
    if not meta["isbn"]:
        m = re.search(r"ISBN(?:-13)?:?\s*([0-9Xx\-]{10,17})", soup.get_text(" ", strip=True))
        if m:
            meta["isbn"] = only_digits(m.group(1))

    return meta

# Asegura columnas base existan en keep
for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]:
    if c not in keep.columns:
        keep[c] = ""

In [26]:
# --------- 2) Enriquecer en lote ---------
print("\n=== ENRIQUECIMIENTO ===")
rows_meta, t0 = [], datetime.now()

for i, url in enumerate(to_enrich["url_producto"], start=1):
    meta = extract_meta_from_product(url)
    meta["url_producto"] = url
    rows_meta.append(meta)

    if i % 50 == 0:
        print(f"  - enriquecidos: {i}/{len(to_enrich)}")
    time.sleep(random.uniform(*ENRICH_SLEEP))

print("Tiempo enriquecimiento:", datetime.now()-t0)

# --------- 3) Merge robusto por URL ----------
if rows_meta:
    upd = pd.DataFrame(rows_meta)

    # prefijo para evitar choques
    rename_map = {c: f"enr_{c}" for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]}
    upd = upd.rename(columns=rename_map)

    keep = keep.merge(
        upd[["url_producto"] + list(rename_map.values())],
        on="url_producto",
        how="left"
    )

    # completa si el campo original está vacío
    for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]:
        src = f"enr_{c}"
        if src in keep.columns:
            keep[c] = keep[c].astype(str)
            keep[c] = keep[c].where(keep[c].str.strip().ne(""), keep[src])
            keep.drop(columns=[src], inplace=True, errors="ignore")

    # normaliza nulos
    for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]:
        keep[c] = keep[c].fillna("").astype(str)

    print("[OK] Integración de enriquecimiento completada.")
else:
    print("No hubo filas enriquecidas (revisa selectores o URLs).")


=== ENRIQUECIMIENTO ===
  - enriquecidos: 50/11174
  - enriquecidos: 100/11174
  - enriquecidos: 150/11174
  - enriquecidos: 200/11174
  - enriquecidos: 250/11174
  - enriquecidos: 300/11174
  - enriquecidos: 350/11174
  - enriquecidos: 400/11174
  - enriquecidos: 450/11174
  - enriquecidos: 500/11174
  - enriquecidos: 550/11174
  - enriquecidos: 600/11174
  - enriquecidos: 650/11174
  - enriquecidos: 700/11174
  - enriquecidos: 750/11174
  - enriquecidos: 800/11174
  - enriquecidos: 850/11174
  - enriquecidos: 900/11174
  - enriquecidos: 950/11174
  - enriquecidos: 1000/11174
  - enriquecidos: 1050/11174
  - enriquecidos: 1100/11174
  - enriquecidos: 1150/11174
  - enriquecidos: 1200/11174
  - enriquecidos: 1250/11174
  - enriquecidos: 1300/11174
  - enriquecidos: 1350/11174
  - enriquecidos: 1400/11174
  - enriquecidos: 1450/11174
  - enriquecidos: 1500/11174
  - enriquecidos: 1550/11174
  - enriquecidos: 1600/11174
  - enriquecidos: 1650/11174
  - enriquecidos: 1700/11174
  - enriq

In [27]:
# --------- 4) Guardar y diagnóstico breve ----------
ENRICHED_OUT = os.path.join(LOCAL_DIR, "crisol_snapshot_enriched.csv")
keep.to_csv(ENRICHED_OUT, index=False, encoding="utf-8-sig")
print("[OK] Guardado enriched:", ENRICHED_OUT)

print("\n=== DIAGNÓSTICO DE ENRIQUECIMIENTO ===")
for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id"]:
    filled = keep[c].astype(str).str.strip().ne("").mean()*100
    print(f"{c:12s}: {filled:4.1f}% lleno")

# --- IDs estables y llave canónica ---
import hashlib
import pandas as pd

# 0) columnas que usamos para la clave
for c in ["isbn","sku","product_id","url_norm"]:
    if c not in keep.columns:
        keep[c] = ""

VENDOR = "crisol"  # para crisol usa "crisol"

def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

# 1) product_id_stable: hash estable por vendor + url_norm
keep["product_id_stable"] = (
    VENDOR + "::" + keep["url_norm"].astype(str)
).apply(sha1)

# 2) product_key: prioridad isbn -> sku -> product_id -> product_id_stable
#    (vectorizado, sin .where en cascada)
key_matrix = (
    keep[["isbn","sku","product_id","product_id_stable"]]
    .astype(str).apply(lambda s: s.str.strip())
    .replace({"": pd.NA})
)
keep["product_key"] = key_matrix.bfill(axis=1).iloc[:, 0]

# 3) vendor (útil si luego unes SBS + Crisol)
keep["vendor"] = VENDOR

# (opcional) chequeos rápidos
print("\n=== CHEQUEO KEYS ===")
print("product_key vacíos:", keep["product_key"].isna().sum())
print("product_key únicos:", keep["product_key"].nunique(), "de", len(keep))

# versión final con columnas en orden
FINAL_COLS = [
    "fecha_scraping","titulo","autor","precio_actual","precio_anterior",
    "url_producto","categoria","categorias_join","url_norm",
    "isbn","sku","editorial","edicion","anio_edicion","paginas","product_id",
    "product_id_stable","product_key","vendor"
]
final_path = os.path.join(LOCAL_DIR, "crisol_final_enriched.csv")
keep[FINAL_COLS].to_csv(final_path, index=False, encoding="utf-8-sig")
print("\n[OK] Dataset FINAL listo:")
print(" - Ruta :", final_path)
print(" - Filas:", len(keep))
print(" - Cols :", FINAL_COLS)

[OK] Guardado enriched: /srv/bigdata/crisol_snapshot_enriched.csv

=== DIAGNÓSTICO DE ENRIQUECIMIENTO ===
autor       : 79.8% lleno
isbn        : 99.8% lleno
sku         : 79.8% lleno
editorial   : 88.2% lleno
edicion     : 76.6% lleno
anio_edicion: 76.6% lleno
paginas     : 78.3% lleno
product_id  : 94.4% lleno

=== CHEQUEO KEYS ===
product_key vacíos: 0
product_key únicos: 11179 de 11179

[OK] Dataset FINAL listo:
 - Ruta : /srv/bigdata/crisol_final_enriched.csv
 - Filas: 11179
 - Cols : ['fecha_scraping', 'titulo', 'autor', 'precio_actual', 'precio_anterior', 'url_producto', 'categoria', 'categorias_join', 'url_norm', 'isbn', 'sku', 'editorial', 'edicion', 'anio_edicion', 'paginas', 'product_id', 'product_id_stable', 'product_key', 'vendor']


## Validaciones

In [28]:
print("\n=== VALIDACIONES ===")

# 1) Precios a numérico
for c in ["precio_actual", "precio_anterior"]:
    keep[c] = pd.to_numeric(keep[c], errors="coerce")

# 2) Campos enriquecidos a tipos útiles (si existen)
for c in ["edicion", "anio_edicion", "paginas"]:
    if c in keep.columns:
        keep[c] = pd.to_numeric(keep[c], errors="coerce").astype("Int64")

# 3) Rango simple
n_out = (keep["precio_actual"].fillna(0) < 0).sum()
print("Precios negativos:", n_out)

# 4) ID estable (hash de url_norm) – NO reemplaza product_id,
#    solo sirve como ID estable cuando product_id viene vacío.
import hashlib
keep["product_id_stable"] = (
    keep["url_norm"]
      .astype(str)
      .apply(lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest())
)

# 5) Preview rápido
print(keep.dtypes)
print("Preview enriched/dedup:")
keep.head(3)


=== VALIDACIONES ===
Precios negativos: 0
fecha_scraping        object
titulo                object
autor                 object
precio_actual        float64
precio_anterior      float64
url_producto          object
categoria             object
categorias_join       object
url_norm              object
isbn                  object
sku                   object
editorial             object
edicion                Int64
anio_edicion           Int64
paginas                Int64
product_id            object
product_id_stable     object
product_key           object
vendor                object
dtype: object
Preview enriched/dedup:


,fecha_scraping,titulo,autor,precio_actual,precio_anterior,url_producto,categoria,categorias_join,url_norm,isbn,sku,editorial,edicion,anio_edicion,paginas,product_id,product_id_stable,product_key,vendor
0,2025-10-20,12 Reglas para Vivir. Un Antídoto al Caos,"Peterson, Jordan B.",51.92,64.9,https://www.crisol.com.pe/12-reglas-para-vivir...,No Ficcion / Humanidades,No Ficcion / Humanidades | No Ficcion / Humani...,https://www.crisol.com.pe/12-reglas-para-vivir...,9786125027047,"Peterson, Jordan B.",BOOKET,20210,20210,512,444147,ef86653a1a249e9f9cd92d9736067cd105dadeed,9786125027047,crisol
1,2025-10-20,2025 Family Desk Pad and Wall Calendar,,49.01,NaN,https://www.crisol.com.pe/2025-family-desk-pad...,Smart Play / Complementos / Agendas Y Libretas,Smart Play / Complementos / Agendas Y Libretas,https://www.crisol.com.pe/2025-family-desk-pad...,9781441342379,,PETER PAUPER PRESS,<NA>,<NA>,<NA>,1188571,d3983fec7e581004da697d3ac1b5e9fb32f27740,9781441342379,crisol
2,2025-10-20,24 Colores Súper Intensos & Sacapuntas Artesco,,14.90,NaN,https://www.crisol.com.pe/24-colores-super-int...,Smart Play / Complementos / Utiles De Escritorio,Smart Play / Complementos / Utiles De Escritorio,https://www.crisol.com.pe/24-colores-super-int...,7750082003620,,ARTESCO,<NA>,<NA>,<NA>,1127955,21104b2e15195cd332e302455225cd120858f0b7,7750082003620,crisol


## Guardar Artefactos

In [29]:
# === MANIFEST ===
import json, os, hashlib
from datetime import datetime

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

# Rutas que ya existen por pasos previos
RAW_SNAPSHOT   = SNAPSHOT_OUT                     # p.ej. /srv/bigdata/crisol_precios_allcats_YYYYMMDD_HHMMSS.csv
DEDUP_CSV      = DEDUP_OUT                        # /srv/bigdata/crisol_snapshot_dedup.csv
ENRICHED_CSV   = final_path                       # /srv/bigdata/crisol_final_enriched.csv
INGEST_DATE    = RAW_DAY                          # yyyy-mm-dd
VENDOR         = "crisol"

# Métricas rápidas
fill_rates = {
    c: float(keep[c].astype(str).str.strip().ne("").mean()*100)
    for c in ["autor","isbn","sku","editorial","edicion","anio_edicion","paginas","product_id","product_id_stable","product_key"]
    if c in keep.columns
}

price_stats = {}
for c in ["precio_actual","precio_anterior"]:
    if c in keep.columns:
        s = pd.to_numeric(keep[c], errors="coerce")
        price_stats[c] = {
            "count": int(s.notna().sum()),
            "min": float(s.min(skipna=True)) if s.notna().any() else None,
            "p50": float(s.quantile(0.5))     if s.notna().any() else None,
            "p90": float(s.quantile(0.9))     if s.notna().any() else None,
            "max": float(s.max(skipna=True)) if s.notna().any() else None,
        }

manifest = {
    "version": "1.0",
    "vendor": VENDOR,
    "source": "crisol.com.pe",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "ingest_date": INGEST_DATE,
    "files": {
        "raw_snapshot":   {"path": RAW_SNAPSHOT, "sha256": sha256(RAW_SNAPSHOT)},
        "dedup":          {"path": DEDUP_CSV,    "sha256": sha256(DEDUP_CSV)},
        "final_enriched": {"path": ENRICHED_CSV, "sha256": sha256(ENRICHED_CSV)},
    },
    "shape": {
        "rows_raw":   int(len(df)),
        "rows_dedup": int(len(keep)),     # tras dedup y antes de filtros extra (si los hubo)
        "rows_final": int(len(keep)),     # igual si no filtraste después
        "cols_final": list(keep.columns),
    },
    "uniques": {
        "url_norm": int(keep["url_norm"].nunique()) if "url_norm" in keep.columns else None,
        "product_key": int(keep["product_key"].nunique()) if "product_key" in keep.columns else None,
    },
    "fill_rates_pct": fill_rates,
    "price_stats": price_stats,
    "top_categories": keep["categoria"].value_counts().head(10).to_dict() if "categoria" in keep.columns else {},
}

MANIFEST_OUT = os.path.join(LOCAL_DIR, "crisol_manifest.json")
with open(MANIFEST_OUT, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("[OK] Guardado manifest:", MANIFEST_OUT)

[OK] Guardado manifest: /srv/bigdata/crisol_manifest.json


## Subir a HDFS

In [30]:
import os, subprocess, json, hashlib
from pathlib import Path

artefactos = {
    "raw_snapshot":   os.path.abspath(SNAPSHOT_OUT),
    "dedup_csv":      os.path.abspath(DEDUP_OUT),
    "final_enriched": os.path.abspath(final_path),
    "manifest_json":  os.path.abspath(MANIFEST_OUT),
}

# 1) Validaciones locales
for k, p in artefactos.items():
    assert os.path.exists(p), f"[faltante] {k}: {p}"
print("[OK] Artefactos locales encontrados:")
for k, p in artefactos.items():
    print(f"  - {k}: {p} ({Path(p).stat().st_size/1024/1024:.2f} MB)")

# 2) Carpeta destino (Raw Zone particionada por día)
HDFS_DIR = f"/data/raw/crisol/ingest_date={RAW_DAY}/"

def run(*args, check=True):
    return subprocess.run(list(args), check=check, text=True, capture_output=True)

# 3) Crear destino y subir
print("\n[HDFS] Creando destino:", HDFS_DIR)
run("hdfs","dfs","-mkdir","-p", HDFS_DIR)

print("[HDFS] Subiendo artefactos…")
for k, p in artefactos.items():
    print(f"  -> put {k}")
    run("hdfs","dfs","-put","-f", p, HDFS_DIR)

# 4) Listado y verificación básica
print("\n[HDFS] Contenido final:")
ls_out = run("hdfs","dfs","-ls","-h", HDFS_DIR, check=False)
print(ls_out.stdout or ls_out.stderr)

# 5) Replicación y salud (rápido)
print("[HDFS] Replicación por archivo (debe ser 2):")
for k, p in artefactos.items():
    fn = os.path.basename(p)
    try:
        stat_r = run("hdfs","dfs","-stat","%r", HDFS_DIR + fn, check=False)
        print(f"  - {fn}: {stat_r.stdout.strip() or stat_r.stderr.strip()}")
    except Exception as e:
        print(f"  - {fn}: no se pudo consultar (%s)" % e)

# Comprobación de fsck resumida
print("\n[HDFS] FSCK (resumen de replicación):")
fsck = run("hdfs","fsck", HDFS_DIR, "-files","-blocks","-racks", check=False)
for line in fsck.stdout.splitlines():
    if any(kw in line for kw in ["Over-replicated","Under replicated","replicas"]):
        print(" ", line)
print("Listo.")

[OK] Artefactos locales encontrados:
  - raw_snapshot: /srv/bigdata/crisol_precios_allcats_20251020_203525.csv (2.66 MB)
  - dedup_csv: /srv/bigdata/crisol_snapshot_dedup.csv (2.82 MB)
  - final_enriched: /srv/bigdata/crisol_final_enriched.csv (4.29 MB)
  - manifest_json: /srv/bigdata/crisol_manifest.json (0.00 MB)

[HDFS] Creando destino: /data/raw/crisol/ingest_date=2025-10-20/
[HDFS] Subiendo artefactos…
  -> put raw_snapshot
  -> put dedup_csv
  -> put final_enriched
  -> put manifest_json

[HDFS] Contenido final:
Found 4 items
-rw-rw-r--+  2 bigdata bd      4.3 M 2025-10-21 07:41 /data/raw/crisol/ingest_date=2025-10-20/crisol_final_enriched.csv
-rw-rw-r--+  2 bigdata bd      2.1 K 2025-10-21 07:41 /data/raw/crisol/ingest_date=2025-10-20/crisol_manifest.json
-rw-rw-r--+  2 bigdata bd      2.7 M 2025-10-21 07:41 /data/raw/crisol/ingest_date=2025-10-20/crisol_precios_allcats_20251020_203525.csv
-rw-rw-r--+  2 bigdata bd      2.8 M 2025-10-21 07:41 /data/raw/crisol/ingest_date=2025-10

## Processed

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, to_date, col
from pyspark.sql.types import DoubleType, StringType

spark = (
    SparkSession.builder
      .master("spark://vm1-master:7077")
      .appName("productos-crisol-processed")
      .config("spark.sql.shuffle.partitions", "8")
      .config("spark.sql.parquet.compression.codec", "snappy")
      .getOrCreate()
)

INGEST_DAY = "2025-10-20"  # <-- ajusta a tu partición
VENDOR = "crisol"

# ---------- A) SILVER: dedup ----------
src_silver = f"hdfs:///data/raw/crisol/ingest_date={INGEST_DAY}/crisol_snapshot_dedup.csv"
df_silver = spark.read.option("header", True).csv(src_silver)

# tipado mínimo + metadata partición
for c in ["precio_actual","precio_anterior"]:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, col(c).cast(DoubleType()))

df_silver = (df_silver
    .withColumn("ingest_date", to_date(lit(INGEST_DAY)))
    .withColumn("vendor", lit(VENDOR).cast(StringType()))
)

out_silver = "hdfs:///data/processed/parquet/productos_crisol_silver"
# Para datasets crecientes usa .mode("append"); como es one-shot, "overwrite" está OK.
(df_silver
    # .repartition(8, "ingest_date")     # mejor que coalesce(1) en conjuntos grandes
    .write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(out_silver)
)

print("[OK] Silver ->", out_silver)

# ---------- B) GOLD: enriquecido ----------
src_gold = f"hdfs:///data/raw/crisol/ingest_date={INGEST_DAY}/crisol_final_enriched.csv"
df_gold = spark.read.option("header", True).csv(src_gold)

for c in ["precio_actual","precio_anterior"]:
    if c in df_gold.columns:
        df_gold = df_gold.withColumn(c, col(c).cast(DoubleType()))

df_gold = (df_gold
    .withColumn("ingest_date", to_date(lit(INGEST_DAY)))
    .withColumn("vendor", lit(VENDOR).cast(StringType()))
)

# (opcional) normaliza columnas esperadas en gold
expected = [
    "fecha_scraping","titulo","autor","precio_actual","precio_anterior",
    "url_producto","categoria","categorias_join","url_norm",
    "isbn","sku","editorial","edicion","anio_edicion","paginas","product_id",
    "product_id_stable","product_key","vendor","ingest_date"
]
df_gold = df_gold.select([c for c in expected if c in df_gold.columns])

out_gold = "hdfs:///data/processed/parquet/productos_crisol_gold"
(df_gold
    # .repartition(8, "ingest_date")
    .write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(out_gold)
)

print("[OK] Gold   ->", out_gold)

25/10/21 07:45:28 WARN Utils: Your hostname, Proyecto-BigData resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
25/10/21 07:45:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/21 07:45:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

[OK] Silver -> hdfs:///data/processed/parquet/productos_crisol_silver


[OK] Gold   -> hdfs:///data/processed/parquet/productos_crisol_gold


In [32]:
spark.stop()